# Predict appointment attendance in the next 30 days

Which patients will attend at least one appointment in the next 30 days? Train a local Kumo Engine model on generated synthetic relational data, then score newly added records.

Use a Linux notebook kernel with supported NVIDIA GPU compute and local SSD scratch. Complete [Linux deployment setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/linux-machine-deployment) and consult [GPU sizing](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/gpu-sizing) for your machine.
This example follows [the binary classification workflow](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/code-examples/linux-deployment/binary-classification).


## Install

Install the Engine SDK and the data libraries used by this notebook.


In [1]:
%pip install kumo-engine pandas numpy pyarrow


Note: you may need to restart the kernel to use updated packages.


## Import the local Engine APIs

Use the local Engine graph, query, and trainer APIs in the prepared notebook kernel.


In [2]:
import json
import os
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import pandas as pd
from IPython.display import display
import kumo.engine as kumo
from kumo.engine.trainer.output_config import ModelOutputConfig, PredictionOutputConfig, TrainingTableOutputConfig
from kumoapi.graph import Edge
from kumoapi.jobs import MetadataField, PredictionArtifactType
from kumoapi.model_plan import RunMode, TrainingTableGenerationPlan
from kumoapi.typing import Dtype, Stype


/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Set the example parameters

Set `KUMO_SCRATCH` to an existing writable directory on local SSD before running. Each run gets its own scratch and retained output directories. Times are timezone-naive UTC.
Leave `SMOKE_MODE=False` for the FAST tutorial plan; smoke mode caps training at one epoch and does not measure model quality.


In [3]:
SCRATCH_PARENT = Path(os.environ["KUMO_SCRATCH"]).expanduser().resolve()
OUTPUT_PARENT = Path.cwd() / "kumo_example_runs"
RANDOM_SEED = 42
AS_OF = pd.Timestamp("2025-01-01")
SMOKE_MODE = False
DECISION_THRESHOLD = 0.5

if not SCRATCH_PARENT.is_dir():
    raise ValueError("KUMO_SCRATCH must point to an existing local SSD directory.")
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
run_dir = Path(mkdtemp(prefix="synthetic_appointment_attendance-", dir=OUTPUT_PARENT))
scratch_dir = Path(mkdtemp(prefix="synthetic_appointment_attendance-", dir=SCRATCH_PARENT))
source_dir = run_dir / "source"
output_dir = run_dir / "output"
source_dir.mkdir()
output_dir.mkdir()
kumo.configure(scratch_dir=scratch_dir)
destination = kumo.FileConnector(root_dir=str(output_dir))
print("Retained run:", run_dir)
print("Scratch:", scratch_dir)


Retained run: /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/kumo_example_runs/synthetic_appointment_attendance-8v8xdhtt
Scratch: /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/scratch/synthetic_appointment_attendance-u_9kcl37


## Generate synthetic example data

Create synthetic patients, clinics, and attended visits for capacity planning. The generator varies visit frequency by patient and clinic preference; it includes no diagnoses or real patient records.
Run the next cell to generate the history and a separate batch reserved for scoring. You can skim the generation code; the Kumo workflow starts below.


In [4]:
rng = np.random.default_rng(RANDOM_SEED)
clinics = pd.DataFrame({
    "clinic_id": np.arange(1, 13, dtype=np.int64),
    "service": np.tile(["general", "physiotherapy", "dental"], 4),
    "region": np.repeat(["north", "south", "east", "west"], 3),
})

def make_patients_and_visits(first_id, count, start, end, first_visit):
    ids = np.arange(first_id, first_id + count, dtype=np.int64)
    patients = pd.DataFrame({"patient_id": ids,
        "region": rng.choice(["north", "south", "east", "west"], count),
        "booking_channel": rng.choice(["phone", "portal"], count)})
    records = []
    days = (pd.Timestamp(end) - pd.Timestamp(start)).days
    for patient_id in ids:
        rate = rng.choice([0.008, 0.025, 0.06], p=[0.4, 0.4, 0.2])
        preferred_clinic = int(rng.choice(clinics.clinic_id))
        for day in np.flatnonzero(rng.random(days) < rate):
            clinic = preferred_clinic if rng.random() < 0.8 else int(rng.choice(clinics.clinic_id))
            records.append((first_visit + len(records), patient_id, clinic,
                            pd.Timestamp(start) + pd.Timedelta(days=int(day), hours=12)))
    visits = pd.DataFrame(records, columns=["visit_id", "patient_id", "clinic_id", "attended_at"])
    return patients, visits

patients, visits = make_patients_and_visits(1, 600, "2024-01-01", AS_OF, 1)
new_patients, new_visits = make_patients_and_visits(
    601, 40, AS_OF - pd.Timedelta(days=60), AS_OF, int(visits.visit_id.max()) + 1)
assert visits.visit_id.is_unique
assert visits.patient_id.isin(patients.patient_id).all()
assert visits.clinic_id.isin(clinics.clinic_id).all()
assert visits.attended_at.lt(AS_OF).all() and new_visits.attended_at.lt(AS_OF).all()
for name, frame in {"patients": patients, "clinics": clinics, "visits": visits}.items():
    (source_dir / name).mkdir()
    frame.to_parquet(source_dir / name / "history.parquet", index=False)
display(visits.head())
print("Patients:", len(patients), "Attended visits:", len(visits))


/tmp/ipykernel_216916/2308271359.py:21: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  pd.Timestamp(start) + pd.Timedelta(days=int(day), hours=12)))
/tmp/ipykernel_216916/2308271359.py:27: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  601, 40, AS_OF - pd.Timedelta(days=60), AS_OF, int(visits.visit_id.max()) + 1)


,visit_id,patient_id,clinic_id,attended_at
0,1,1,12,2024-02-14 12:00:00
1,2,1,12,2024-03-21 12:00:00
2,3,1,12,2024-05-18 12:00:00
3,4,1,12,2024-05-21 12:00:00
4,5,1,12,2024-06-12 12:00:00


Patients: 600 Attended visits: 5525


## Build the graph

Connect attended visits to patients and clinics. Visit timestamps let Kumo use past attendance as context and later visits as labels; only stable intake attributes appear on patient rows.
See [tables and relationships](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs).


In [5]:
files = kumo.FileConnector(root_dir=str(source_dir))
patient_table = kumo.Table.from_source_table(files["patients"], primary_key="patient_id")
for column in ("region", "booking_channel"):
    patient_table = patient_table.with_column(column, stype=Stype.categorical)
clinic_table = kumo.Table.from_source_table(files["clinics"], primary_key="clinic_id")
for column in ("service", "region"):
    clinic_table = clinic_table.with_column(column, stype=Stype.categorical)
visit_table = (kumo.Table.from_source_table(
    files["visits"], primary_key="visit_id", time_column="attended_at")
    .with_column("attended_at", dtype=Dtype.date)
    .with_column("patient_id", stype=Stype.ID)
    .with_column("clinic_id", stype=Stype.ID))
graph = kumo.Graph(tables={"patients": patient_table, "clinics": clinic_table, "visits": visit_table},
    edges=[Edge("visits", "patient_id", "patients"), Edge("visits", "clinic_id", "clinics")])
graph.validate()


## Inspect the graph

Inspect the actual graph's relationships before training. The [graph visualization](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) call is a commented preview because the supported package baseline does not expose it. The relationship table below is executable.


In [6]:
# Preview only: graph.visualize(show_cols=True)
display(pd.DataFrame([{"source": edge.src_table, "foreign_key": edge.fkey,
                       "destination": edge.dst_table} for edge in graph.edges]))


,source,foreign_key,destination
0,visits,patient_id,patients
1,visits,clinic_id,clinics


## Materialize the graph

Build a snapshot of the historical source files for training.


In [7]:
snapshot = graph.snapshot()
print("Training snapshot:", snapshot.id)


ingest: table started phase="encode" table=patients
ingest: table producers complete phase="encode" table=patients rows=600 elapsed_ms=4
ingest: Ingestion stage progress phase=Encode table=patients shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=clinics
ingest: table producers complete phase="encode" table=clinics rows=12 elapsed_ms=1
ingest: Ingestion stage progress phase=Encode table=clinics shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=visits
ingest: table producers complete phase="encode" table=visits rows=5525 elapsed_ms=14
ingest: Ingestion stage progress phase=Encode table=visits shard=None progress_pct=100 current=1 total=1
ingest: encode stage complete stage="encode" elapsed_ms=37 elapsed=37ms context_ms=0
ingest: Ingestion stage progress phase=PkMap table=clinics shard=None progress_pct=100 current=1 total=1
ingest: Ingestion stage progress phase=PkMap table=patients shard=None progress_pct=10

Training snapshot: fe1b85758a6645db7ad19d1453f15edeccc9abb1dcfe232cc358eba5abbbf3d0@e92e6354d84f064b


## Define the prediction task

At each historical anchor, a positive label means at least one attended visit in the interval `(anchor, anchor + 30 days]`. The output is the probability of at least one attended visit, not a visit count or the outcome of a particular booked appointment.
Validate the [predictive query](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries) before generating labels.


In [8]:
QUERY = 'PREDICT COUNT(visits.*, 0, 30, days) > 0 FOR EACH patients.patient_id'
query = kumo.PredictiveQuery(graph, QUERY)
query.validate()
assert query.get_task_type() == 'binary_classification'
print(QUERY)
print("Task:", query.get_task_type())


PREDICT COUNT(visits.*, 0, 30, days) > 0 FOR EACH patients.patient_id
Task: binary_classification


## Generate training rows and inspect the splits

Use an approximate chronological 70/15/15 split so validation and test follow training. Use the most recent 300 days of history, with anchors 30 days apart. ApproxDateOffsetSplit separates label windows across splits; the latest anchor leaves 30 days of observed outcomes. Inspect the actual split sizes below.
Export the [training table](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training-tables) to inspect its actual splits: `0` is training, `1` validation, and `2` test. `ENTITY1` identifies the entity, `TIMESTAMP` its anchor, and `TARGET1` the outcome.


In [9]:
generation_plan = TrainingTableGenerationPlan(
    split="ApproxDateOffsetSplit([0.7, 0.15, 0.15])",
    train_start_offset=300, train_end_offset=0, timeframe_step=30,
)
training_table = query.generate_training_table(snapshot, plan=generation_plan)
training_table.export(TrainingTableOutputConfig(connector=destination, table_name="training_rows/"))
training_rows = pd.read_parquet(output_dir / "training_rows")
display(training_rows.head(8))
display(training_rows.groupby("SPLIT").agg(rows=("ENTITY1", "size"),
    first_anchor=("TIMESTAMP", "min"), last_anchor=("TIMESTAMP", "max")))
assert set(training_rows.SPLIT) == {0, 1, 2}


,ENTITY1,TIMESTAMP,TARGET1,SPLIT
0,169,2024-03-06 12:00:00,True,0
1,169,2024-04-05 12:00:00,False,0
2,169,2024-05-05 12:00:00,True,0
3,169,2024-06-04 12:00:00,False,0
4,169,2024-07-04 12:00:00,False,0
5,169,2024-08-03 12:00:00,True,0
6,169,2024-09-02 12:00:00,True,0
7,170,2024-03-06 12:00:00,False,0


,rows,first_anchor,last_anchor
SPLIT,,,
0,4200,2024-03-06 12:00:00,2024-09-02 12:00:00
1,1200,2024-10-02 12:00:00,2024-11-01 12:00:00
2,600,2024-12-01 12:00:00,2024-12-01 12:00:00


## Configure the model

Start from the supported [FAST plan](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training). Use AUPRC to select the binary model and report precision and recall alongside it.
Disable refitting to keep validation and test separate from the fitted training population.


In [10]:
plan = query.suggest_model_plan(snapshot, run_mode=RunMode.FAST)
plan.training_job.refit_trainval = False
plan.training_job.refit_full = False

plan.training_job.metrics = ["auprc", "auroc", "precision", "recall"]
plan.training_job.tune_metric = "auprc"
if SMOKE_MODE:
    plan.training_job.num_experiments = 1
    plan.optimization.max_epochs = 1
    print("Smoke mode: pipeline check only")
(output_dir / "model_plan.json").write_text(json.dumps(plan.as_json(), indent=2) + "\n")
print("Tuning metric:", plan.training_job.tune_metric)
print("Maximum epochs:", plan.optimization.max_epochs)


Tuning metric: auprc
Maximum epochs: 8


## Train and retain the model

Fit on the generated training rows and export the complete model bundle for reuse.


In [11]:
trainer = kumo.Trainer(plan)
fit_result = trainer.fit(training_table, snapshot=snapshot,
    export_config=ModelOutputConfig(connector=destination, path="model"))
print("Saved model:", output_dir / "model")


wrote col_stats.json (and kumo_col_stats/) under /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/scratch/synthetic_appointment_attendance-u_9kcl37/snapshots/fe1b85758a6645db7ad19d1453f15edeccc9abb1dcfe232cc358eba5abbbf3d0/e92e6354d84f064bf5ab77170b87d6d1be5e1006958ff7adea524cc104bb9701/train_tables/.tt-bc5e3fbda4f64c41892ad60c3cddd001-fit-7e1e0208db8a46cdbb811c724c9c8fd1.ingested
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/kumo_core/automl/optuna_searcher.py:40: FutureWarning: `warn_independent_sampling` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  sampler=getattr(optuna.samplers, sampler_name)(**kwargs),
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/kumo_core/automl/optuna_searcher.py:40: ExperimentalWarning: Argument ``multivariate`` is an experimental featu

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 0:   0%|          | 0/30 [00:00<?, ?it/s]

/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='min')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(


Epoch 0: 100%|██████████| 30/30 [00:00<00:00, 36.41it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 30/30 [00:00<00:00, 39.53it/s, val_loss=0.629, val_auprc=0.675, val_auroc=0.713, val_precision=0.614, val_recall=0.704, train_loss=0.646, train_auprc=0.658, train_auroc=0.676, train_precision=0.643, train_recall=0.527]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 30/30 [00:00<00:00, 37.76it/s, val_loss=0.638, val_auprc=0.679, val_auroc=0.717, val_precision=0.586, val_recall=0.719, train_loss=0.617, train_auprc=0.698, train_auroc=0.716, train_precision=0.675, train_recall=0.570]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 30/30 [00:00<00:00, 39.33it/s, val_loss=0.637, val_auprc=0.666, val_auroc=0.713, val_precision=0.602, val_recall=0.692, train_loss=0.598, train_auprc=0

... (20 outputs left out)

Restoring states from the checkpoint path at file:///mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/scratch/synthetic_appointment_attendance-u_9kcl37/trainers/bc5e3fbda4f64c41892ad60c3cddd001/runs/bc5e3fbda4f64c41892ad60c3cddd001-fit-7e1e0208db8a46cdbb811c724c9c8fd1/models/bc5e3fbda4f64c41892ad60c3cddd001-fit-7e1e0208db8a46cdbb811c724c9c8fd1/automl/version_1/checkpoint/epoch=4-step=150.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at file:///mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/scratch/synthetic_appointment_attendance-u_9kcl37/trainers/bc5e3fbda4f64c41892ad60c3cddd001/runs/bc5e3fbda4f64c41892ad60c3cddd001-fit-7e1e0208db8a46cdbb811c724c9c8fd1/models/bc5e3fbda4f64c41892ad60c3cddd001-fit-7e1e0208db8a46cdbb811c724c9c8fd1/automl/version_1/checkpoint/epoch=4-step=150.ckpt
/mnt/localssd/kumo-notebook

Testing DataLoader 0:   0%|          | 0/3 [00:00<?, ?it/s]

/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='min')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/torch_geometric/utils/_scatter.py:91: UserWarning: The usage of `scatter(reduce='max')` can be accelerated via the 'torch-scatter' package, but it was not found
  warnings.warn(
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python3.12/site-packages/kumo_core/explain/subgraph_collector.py:190: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[2 2 2 ... 464 464 464]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[index.cpu().numpy(), fkey_name] = dst_pkey[
/mnt/localssd/kumo-notebook-execution-20261106/venv/lib/python

Testing DataLoader 0: 100%|██████████| 3/3 [00:05<00:00,  0.58it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           0.7679685950279236
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Saved model: /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/kumo_example_runs/synthetic_appointment_attendance-8v8xdhtt/output/model


## Evaluate

**Synthetic-data metrics — demonstration only.** AUPRC summarizes how well the model ranks patients with at least one attended visit in the following 30 days. Precision is the fraction of predicted attendees who attend; recall is the fraction of attendees identified.
Choose settings using validation and reserve test results for final assessment. The tutorial threshold is not calibrated for a clinic's planning decisions.


In [12]:
print("Synthetic-data metrics — demonstration only")
display(pd.Series(fit_result.metrics, name="value").rename_axis("metric").to_frame())
(output_dir / "metrics.json").write_text(json.dumps(fit_result.metrics, indent=2, default=float) + "\n")


Synthetic-data metrics — demonstration only


,value
metric,
val_loss,0.633364
val_auprc,0.681512
val_auroc,0.719187
val_precision,0.607256
val_recall,0.702555
test_loss,0.669674
test_auprc,0.657932
test_auroc,0.716724
test_precision,0.552198


555

## Batch predictions

Add 40 new patients and their observed 60-day visit histories, generated earlier with the same process. No visits after the scoring anchor are supplied. Write the new rows as additional Parquet parts in the existing table directories. A new snapshot reads those parts while the training snapshot remains unchanged.


In [13]:
assert set(new_patients.patient_id).isdisjoint(patients.patient_id)
new_patients.to_parquet(source_dir / "patients" / "new.parquet", index=False)
new_visits.to_parquet(source_dir / "visits" / "new.parquet", index=False)
scoring_population = pd.DataFrame({"ENTITY1": new_patients.patient_id, "TIMESTAMP": AS_OF})
assert not any(column.startswith("TARGET") for column in scoring_population)
assert len(scoring_population) == 40 and not scoring_population.duplicated(["ENTITY1", "TIMESTAMP"]).any()
scoring_population.to_parquet(source_dir / "scoring_population.parquet", index=False)
scoring_snapshot = graph.snapshot()
assert scoring_snapshot.id != snapshot.id
display(scoring_population.head())


ingest: table started phase="encode" table=patients
ingest: table producers complete phase="encode" table=patients rows=640 elapsed_ms=4
ingest: Ingestion stage progress phase=Encode table=patients shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=clinics
ingest: table producers complete phase="encode" table=clinics rows=12 elapsed_ms=1
ingest: Ingestion stage progress phase=Encode table=clinics shard=None progress_pct=100 current=1 total=1
ingest: table started phase="encode" table=visits
ingest: table producers complete phase="encode" table=visits rows=5589 elapsed_ms=9
ingest: Ingestion stage progress phase=Encode table=visits shard=None progress_pct=100 current=1 total=1
ingest: encode stage complete stage="encode" elapsed_ms=32 elapsed=32ms context_ms=0
ingest: Ingestion stage progress phase=PkMap table=clinics shard=None progress_pct=100 current=1 total=1
ingest: Ingestion stage progress phase=PkMap table=patients shard=None progress_pct=100

,ENTITY1,TIMESTAMP
0,601,2025-01-01
1,602,2025-01-01
2,603,2025-01-01
3,604,2025-01-01
4,605,2025-01-01


### Prepare the scoring population

Use a [custom prediction table](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/making-predictions) containing only the 40 new entity IDs and their anchor times. This keeps historical training entities out of the scoring batch.


In [14]:
prediction_table = kumo.PredictionTable.from_source_table(
    files["scoring_population"], pq=query, snapshot=scoring_snapshot, plan=generation_plan)
generated_population = pd.read_parquet(prediction_table.table_path)
assert set(generated_population.ENTITY1) == set(scoring_population.ENTITY1)
assert len(generated_population) == len(scoring_population)
display(generated_population.head())


,ENTITY1,TIMESTAMP
0,601,2025-01-01
1,602,2025-01-01
2,603,2025-01-01
3,604,2025-01-01
4,605,2025-01-01


### Score the new records

Run the trained model on the new snapshot and save predictions, including anchor timestamps, as Parquet.


In [15]:
prediction_result = trainer.predict(
    prediction_table, snapshot=scoring_snapshot,
    binary_classification_threshold=DECISION_THRESHOLD,
    output_config=PredictionOutputConfig(connector=destination,
        artifact_type=PredictionArtifactType.PREDICTIONS, table_name="predictions.parquet",
        extra_fields=[MetadataField.ANCHOR_TIMESTAMP]),
)
scores = prediction_result.frame


wrote col_stats.json (and kumo_col_stats/) under /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/scratch/synthetic_appointment_attendance-u_9kcl37/snapshots/fe1b85758a6645db7ad19d1453f15edeccc9abb1dcfe232cc358eba5abbbf3d0/400507ad3b0ae239575308ff7f171a9c3dc3cedd18aceb3eb8fb4ac9721c030d/train_tables/.pred_140f356c75564d7a9c68e7ef0c9e04ea-bc5e3fbda4f64c41892ad60c3cddd001-predict-f56896e2476940d888975c5a67fc5055.ingested
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Restoring states from the checkpoint path at file:///mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/scratch/synthetic_appoin

Predicting DataLoader 0: 100%|██████████| 6/6 [00:00<00:00, 70.59it/s]


### Inspect and verify the results

`ENTITY` is the new record ID and `ANCHOR_TIMESTAMP` its scoring time. Expect one output row per input. `True_PROB` is the probability of the query outcome; `TARGET_PRED` is true only when it exceeds `DECISION_THRESHOLD`. Verify IDs and times together, not row order.


In [16]:
expected = scoring_population.rename(columns={"ENTITY1": "ENTITY", "TIMESTAMP": "ANCHOR_TIMESTAMP"})
expected["ANCHOR_TIMESTAMP"] = pd.to_datetime(expected.ANCHOR_TIMESTAMP)
scores["ANCHOR_TIMESTAMP"] = pd.to_datetime(scores.ANCHOR_TIMESTAMP)

assert not scores.duplicated(["ENTITY", "ANCHOR_TIMESTAMP"]).any()
aligned = expected.merge(scores, on=["ENTITY", "ANCHOR_TIMESTAMP"], how="outer", validate="one_to_one", indicator=True)
assert aligned._merge.eq("both").all()
assert scores.True_PROB.between(0, 1).all()
assert np.isfinite(scores[["False_PROB", "True_PROB"]].to_numpy()).all()
assert np.allclose(scores.False_PROB + scores.True_PROB, 1, atol=1e-6)
assert scores.TARGET_PRED.eq(scores.True_PROB.gt(DECISION_THRESHOLD)).all()
display(scores[["ENTITY", "ANCHOR_TIMESTAMP", "True_PROB", "TARGET_PRED"]].head(10))
pd.testing.assert_frame_equal(scores, pd.read_parquet(output_dir / "predictions.parquet"))
print("Saved predictions:", output_dir / "predictions.parquet")


,ENTITY,ANCHOR_TIMESTAMP,True_PROB,TARGET_PRED
0,601,2025-01-01,0.498812,False
1,607,2025-01-01,0.213550,False
2,613,2025-01-01,0.245777,False
3,619,2025-01-01,0.213341,False
4,625,2025-01-01,0.213822,False
5,631,2025-01-01,0.239782,False
6,637,2025-01-01,0.213503,False
7,602,2025-01-01,0.232997,False
8,608,2025-01-01,0.495338,False
9,614,2025-01-01,0.214010,False


Saved predictions: /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/kumo_example_runs/synthetic_appointment_attendance-8v8xdhtt/output/predictions.parquet


## Retain and reuse

Reload the complete trusted model bundle without fitting again. Keep the source tables and compatible graph schema alongside the bundle; the model does not contain graph data.
A later process must configure scratch, rebuild the graph and snapshot, and prepare its scoring table. See [publish and retrieve models](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/publish-and-retrieve-models).


In [17]:
reloaded_trainer = kumo.Trainer.load(output_dir / "model")
print("Reloaded model from:", output_dir / "model")
print("Retain this run directory:", run_dir)


Reloaded model from: /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/kumo_example_runs/synthetic_appointment_attendance-8v8xdhtt/output/model
Retain this run directory: /mnt/localssd/kumo-notebook-execution-20261106/runs/local_kumo_engine_synthetic_appointment_attendance/20261106T223612Z/kumo_example_runs/synthetic_appointment_attendance-8v8xdhtt


## Cleanup

Keep the generated inputs, model, and predictions under `run_dir`. After scoring and reload checks, optionally remove only this run's scratch directory; rebuild snapshots before further prediction.


In [18]:
import shutil

REMOVE_RUN_SCRATCH = False
if REMOVE_RUN_SCRATCH:
    shutil.rmtree(scratch_dir)
    print("Removed run scratch:", scratch_dir)
